# Multi-Agent Communication Protocol Study — Real API Version

Three-agent pipeline (Planning → Execution → Integration) compared across four
communication protocols (NL / Markdown / JSON / Shared-Memory) on three task
domains (GSM8K / SQuAD / news analysis). Corresponds to the submitted proposal
(Zhang, Zhang, Zhan — March 29, 2026).

All agent/logger/evaluator primitives live in `pipeline.py` so the Streamlit demo
(`app.py`) can reuse them.

## 0. Install & Import

In [1]:
!pip install -q -r requirements.txt

In [2]:
import json, os, sys, time, warnings
from dataclasses import asdict
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from openai import OpenAI
from pipeline import (
    Protocol, TaskDomain,
    Message, CommunicationLogger,
    SharedMemory,
    SYSTEM_PROMPTS, PROTOCOL_INSTRUCTIONS, DOMAIN_MAX_TOKENS,
    llm_call, run_pipeline, RunResult,
    TASK_BUILDERS, EVALUATORS,
    evaluate_math, evaluate_reading, evaluate_news,
    _run_self_tests,
)

_run_self_tests()
print('Imports OK')

pipeline self-tests passed
Imports OK


## 1. API Key Setup

In [3]:
# Load API key: Colab secret → .env file → shell env var → placeholder
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()  # reads .env next to this notebook
    except ImportError:
        pass
    api_key = os.environ.get('OPENAI_API_KEY', 'sk-YOUR-KEY-HERE')

assert api_key and api_key != 'sk-YOUR-KEY-HERE', (
    'OPENAI_API_KEY not found. Create a .env file next to this notebook with '
    'OPENAI_API_KEY=sk-... or export it in the shell before launching Jupyter.'
)

client = OpenAI(api_key=api_key)
MODEL = 'gpt-4o-mini'

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Reply with just OK'}],
    max_tokens=5,
)
print('API connected:', resp.choices[0].message.content.strip())

API connected: OK


## 2. Protocols & Domains

Defined in `pipeline.py`. Four protocols × three domains.

In [4]:
print('Protocols:', [p.value for p in Protocol])
print('Domains:  ', [d.value for d in TaskDomain])

Protocols: ['NL', 'MARKDOWN', 'JSON', 'SHARED_MEMORY']
Domains:   ['MATH', 'READING', 'NEWS']


## 3. Task Data Preparation

- **GSM8K** (HuggingFace) — first 20 items of the test split.
- **SQuAD** (HuggingFace) — first 20 items of the validation split.
- **News** — 10 curated articles with pre-extracted key_facts (the factual
  ground truth for ROUGE scoring).

In [5]:
try:
    from datasets import load_dataset
    gsm8k_full = load_dataset('openai/gsm8k', 'main', split='test')
    GSM8K_SAMPLES = [
        {
            'question': item['question'],
            'answer': item['answer'].split('####')[-1].strip().replace(',', ''),
        }
        for item in list(gsm8k_full)[:20]
    ]
    print(f'Loaded {len(GSM8K_SAMPLES)} GSM8K samples from HuggingFace')
except Exception as e:
    print(f'HuggingFace load failed ({e}), using built-in samples')
    GSM8K_SAMPLES = [
        {'question': "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells every remaining egg at the farmers' market for $2 each. How much does she make every day at the farmers' market?", 'answer': '18'},
        {'question': 'A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?', 'answer': '3'},
        {'question': 'Josh decides to try flipping a house. He buys a house for $80,000 and then puts in $50,000 in repairs. This increased the value of the house by 150%. How much profit did he make?', 'answer': '70000'},
        {'question': 'James decides to run 3 sprints 3 times a week. He runs 60 meters each sprint. How many total meters does he run a week?', 'answer': '540'},
        {'question': 'Every day, Wendi feeds each of her chickens three cups of mixed chicken feed, containing seeds, mealworms and vegetables to help keep them healthy. She gives the chickens their feed in three separate meals. In the morning, she gives her flock of chickens 15 cups of feed. In the afternoon, she gives her chickens another 25 cups of feed. If the final meal of the day requires 2 cups of feed per chicken, how many chickens does Wendi have?', 'answer': '20'},
    ]
print(f'  Sample Q: {GSM8K_SAMPLES[0]["question"][:80]}...')
print(f'  Sample A: {GSM8K_SAMPLES[0]["answer"]}')

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Loaded 20 GSM8K samples from HuggingFace
  Sample Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning an...
  Sample A: 18


In [6]:
try:
    from datasets import load_dataset
    squad_full = load_dataset('rajpurkar/squad', split='validation')
    SQUAD_SAMPLES = [
        {'context': item['context'], 'question': item['question'],
         'answers': item['answers']['text']}
        for item in list(squad_full)[:20]
    ]
    print(f'Loaded {len(SQUAD_SAMPLES)} SQuAD samples from HuggingFace')
except Exception as e:
    print(f'HuggingFace load failed ({e}), using built-in samples')
    SQUAD_SAMPLES = [
        {'context': "Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. The Main Building was designed by Henry Drummond. It is 187 feet tall.", 'question': 'How tall is the Main Building?', 'answers': ['187 feet']},
        {'context': "The city of Denver is located in the western United States. It is the capital and most populous city of Colorado. Denver's population was 715,522 in 2020.", 'question': 'What is the population of Denver?', 'answers': ['715,522', '715522']},
        {'context': 'The Nobel Prize in Physics is awarded annually by the Royal Swedish Academy of Sciences. Albert Einstein received the prize in 1921 for his discovery of the photoelectric effect.', 'question': 'In what year did Einstein receive the Nobel Prize in Physics?', 'answers': ['1921']},
        {'context': "Python was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability with its notable use of significant whitespace.", 'question': 'Who created Python?', 'answers': ['Guido van Rossum']},
        {'context': 'The Amazon River is the largest river by discharge volume of water in the world. It flows through Brazil, Peru, and Colombia. Its length is approximately 6,400 km.', 'question': 'How long is the Amazon River?', 'answers': ['approximately 6,400 km', '6,400 km', '6400 km']},
    ]
print(f'  Sample Q: {SQUAD_SAMPLES[0]["question"]}')
print(f'  Sample A: {SQUAD_SAMPLES[0]["answers"]}')

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Loaded 20 SQuAD samples from HuggingFace
  Sample Q: Which NFL team represented the AFC at Super Bowl 50?
  Sample A: ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']


In [7]:
# Curated news corpus with pre-extracted key_facts.
NEWS_SAMPLES = [
    {
        'title': 'Fed Holds Interest Rates Steady in March 2025',
        'content': ('The Federal Reserve held its benchmark interest rate unchanged at 4.25%-4.50% '
                    'at its March 2025 meeting. Chair Jerome Powell stated that inflation remains '
                    'above the 2% target at 2.8%, and the labor market shows continued strength with '
                    'unemployment at 3.9%. The Fed signaled two potential rate cuts later in 2025 if '
                    'inflation data cooperates. Markets reacted positively, with the S&P 500 rising 0.5%.'),
        'key_facts': ['Fed held rate at 4.25%-4.50%', 'inflation at 2.8%', 'unemployment at 3.9%',
                      'two potential rate cuts in 2025', 'S&P 500 rose 0.5%'],
    },
    {
        'title': 'Apple Reports Record Q1 FY2025 Earnings',
        'content': ('Apple reported Q1 FY2025 revenue of $124.3 billion, up 4% year-over-year. '
                    'iPhone revenue reached $69.1 billion. Services revenue hit a record $26.3 billion. '
                    'EPS was $2.40, beating analyst estimates of $2.35. Gross margin improved to 46.9%. '
                    'CEO Tim Cook highlighted strong growth in emerging markets, particularly India.'),
        'key_facts': ['revenue $124.3 billion', 'iPhone revenue $69.1 billion',
                      'Services revenue $26.3 billion', 'EPS $2.40', 'gross margin 46.9%', 'growth in India'],
    },
    {
        'title': 'Tesla Announces New Affordable EV Model',
        'content': ('Tesla unveiled its new affordable electric vehicle, the Model Q, priced at $25,000. '
                    'The Model Q targets mass-market adoption with a 280-mile range and 0-60 mph in 5.8 seconds. '
                    'Production is slated to begin at Gigafactory Texas in Q3 2025. Tesla stock rose 8% on the '
                    'announcement. Analysts project annual sales of 500,000 units by 2027.'),
        'key_facts': ['Model Q priced at $25,000', '280-mile range', 'production Q3 2025 at Gigafactory Texas',
                      'stock rose 8%', 'projected 500,000 units by 2027'],
    },
    {
        'title': 'Global Chip Shortage Eases as TSMC Expands',
        'content': ('TSMC reported that its Arizona fab is now producing 4nm chips at 90% yield rates. '
                    'Global semiconductor lead times have dropped from 26 weeks to 14 weeks. '
                    'TSMC revenue grew 25% YoY to $22.5 billion in Q1 2025. The company plans $40 billion '
                    'in capex for 2025. Intel and Samsung are also expanding capacity but lag behind.'),
        'key_facts': ['Arizona fab producing 4nm chips at 90% yield', 'lead times dropped from 26 to 14 weeks',
                      'TSMC revenue $22.5 billion up 25% YoY', '$40 billion capex for 2025',
                      'Intel and Samsung lag behind'],
    },
    {
        'title': 'EU Passes Comprehensive AI Regulation Act',
        'content': ("The European Union officially enacted the AI Act, the world's first comprehensive AI law. "
                    'High-risk AI systems must undergo conformity assessments. Companies face fines up to 7% of '
                    'global revenue for violations. The law bans social scoring and real-time biometric surveillance. '
                    'Tech companies have 24 months to comply. The regulation affects all companies serving EU citizens.'),
        'key_facts': ['first comprehensive AI law', 'fines up to 7% of global revenue', 'bans social scoring',
                      'bans real-time biometric surveillance', '24 months to comply'],
    },
    {
        'title': 'SpaceX Starship Completes First Orbital Flight',
        'content': ("SpaceX's Starship completed its first successful orbital flight, spending 90 minutes in orbit "
                    'before splashing down in the Indian Ocean. The Super Heavy booster was caught by the launch tower. '
                    'Starship reached an altitude of 250 km. NASA confirmed Starship as the lunar lander for Artemis III. '
                    'SpaceX plans monthly launches starting Q2 2025. The vehicle carried a 100-ton test payload.'),
        'key_facts': ['90 minutes in orbit', 'splashdown in Indian Ocean', 'booster caught by launch tower',
                      'altitude 250 km', 'lunar lander for Artemis III', '100-ton test payload'],
    },
    {
        'title': 'Amazon Launches Drone Delivery in 10 US Cities',
        'content': ('Amazon expanded its Prime Air drone delivery service to 10 major US cities, including '
                    'New York, Los Angeles, and Chicago. Deliveries under 5 pounds arrive within 30 minutes. '
                    'The MK30 drone has a 15-mile delivery radius and operates in light rain. FAA granted '
                    'beyond-visual-line-of-sight approval. Amazon targets 500 million drone deliveries annually by 2028.'),
        'key_facts': ['10 US cities including NY LA Chicago', 'under 5 pounds within 30 minutes',
                      'MK30 drone 15-mile radius', 'FAA beyond-visual-line-of-sight approval',
                      '500 million deliveries by 2028'],
    },
    {
        'title': 'OpenAI Releases GPT-5 with Reasoning Capabilities',
        'content': ('OpenAI released GPT-5, claiming 40% improvement on graduate-level reasoning benchmarks. '
                    'The model scores 92% on the MATH benchmark and 88% on GPQA. API pricing is $15 per million '
                    'input tokens and $60 per million output tokens. GPT-5 supports 256K context window. '
                    'Early users report significant improvements in code generation and scientific reasoning.'),
        'key_facts': ['40% improvement on reasoning benchmarks', '92% on MATH benchmark', '88% on GPQA',
                      '$15/1M input $60/1M output tokens', '256K context window'],
    },
    {
        'title': 'Japan Earthquake Triggers Tsunami Warning',
        'content': ('A 7.4 magnitude earthquake struck off the coast of Hokkaido, Japan at 3:42 AM local time. '
                    'A tsunami warning was issued for coastal areas within 300 km. Waves up to 3 meters were '
                    'observed. 14 people were injured and approximately 50,000 residents were evacuated. '
                    'The Shinkansen bullet train service was suspended for 6 hours. No nuclear plant damage was reported.'),
        'key_facts': ['7.4 magnitude earthquake', 'off coast of Hokkaido', 'tsunami warning 300 km radius',
                      'waves up to 3 meters', '14 injured 50000 evacuated',
                      'Shinkansen suspended 6 hours', 'no nuclear plant damage'],
    },
    {
        'title': 'Global Carbon Emissions Decline for First Time',
        'content': ('Global CO2 emissions fell 2.1% in 2024, the first annual decline not caused by economic recession. '
                    'Renewable energy now accounts for 35% of global electricity generation. Solar capacity additions '
                    'reached 450 GW, a record. China led with 180 GW of new solar. Coal power generation dropped 5% '
                    'globally. The IEA projects emissions could fall another 3% in 2025 if trends continue.'),
        'key_facts': ['CO2 emissions fell 2.1% in 2024', 'renewables 35% of electricity',
                      'solar capacity 450 GW record', 'China 180 GW new solar', 'coal power dropped 5%',
                      'IEA projects 3% further decline in 2025'],
    },
]
print(f'Loaded {len(NEWS_SAMPLES)} news articles')
for i, n in enumerate(NEWS_SAMPLES):
    print(f'  [{i}] {n["title"]}  ({len(n["key_facts"])} key facts)')

Loaded 10 news articles
  [0] Fed Holds Interest Rates Steady in March 2025  (5 key facts)
  [1] Apple Reports Record Q1 FY2025 Earnings  (6 key facts)
  [2] Tesla Announces New Affordable EV Model  (5 key facts)
  [3] Global Chip Shortage Eases as TSMC Expands  (5 key facts)
  [4] EU Passes Comprehensive AI Regulation Act  (5 key facts)
  [5] SpaceX Starship Completes First Orbital Flight  (6 key facts)
  [6] Amazon Launches Drone Delivery in 10 US Cities  (5 key facts)
  [7] OpenAI Releases GPT-5 with Reasoning Capabilities  (5 key facts)
  [8] Japan Earthquake Triggers Tsunami Warning  (7 key facts)
  [9] Global Carbon Emissions Decline for First Time  (6 key facts)


## 4. Cost Estimate

In [8]:
N_MATH    = min(10, len(GSM8K_SAMPLES))
N_READING = min(10, len(SQUAD_SAMPLES))
N_NEWS    = min(10, len(NEWS_SAMPLES))
N_REPS    = 3
N_TOTAL_RUNS = len(list(Protocol)) * (N_MATH + N_READING + N_NEWS) * N_REPS

est_in  = N_TOTAL_RUNS * 3 * 350
est_out = N_TOTAL_RUNS * 3 * 180
est_usd = est_in / 1e6 * 0.15 + est_out / 1e6 * 0.60

print(f'Samples:       Math={N_MATH}, Reading={N_READING}, News={N_NEWS}')
print(f'Protocols:     {len(list(Protocol))} ({", ".join(p.value for p in Protocol)})')
print(f'Reps/sample:   {N_REPS}')
print(f'Total runs:    {N_TOTAL_RUNS}')
print(f'Est. tokens:   {est_in+est_out:,}')
print(f'Est. cost:     ~${est_usd:.3f} USD')

Samples:       Math=10, Reading=10, News=10
Protocols:     4 (NL, MARKDOWN, JSON, SHARED_MEMORY)
Reps/sample:   3
Total runs:    360
Est. tokens:   572,400
Est. cost:     ~$0.173 USD


## 5. Run Full Experiment Grid

For each (protocol, domain, sample, rep) combination: run the three-agent
pipeline, record the `RunResult` summary, and append every inter-agent message
to `results/results_messages.jsonl` for later case-study / error analysis.

In [9]:
DOMAIN_SAMPLES = {
    TaskDomain.MATH:    GSM8K_SAMPLES[:N_MATH],
    TaskDomain.READING: SQUAD_SAMPLES[:N_READING],
    TaskDomain.NEWS:    NEWS_SAMPLES[:N_NEWS],
}

os.makedirs('results', exist_ok=True)
MESSAGES_PATH = 'results/results_messages.jsonl'

results = []
messages_fp = open(MESSAGES_PATH, 'w', encoding='utf-8')
done = 0
errors = 0

for protocol in Protocol:
    for domain in TaskDomain:
        for idx, sample in enumerate(DOMAIN_SAMPLES[domain]):
            for rep in range(N_REPS):
                try:
                    result, msgs = run_pipeline(
                        protocol, domain, sample, idx,
                        client=client, model=MODEL, seed=rep,
                    )
                    results.append(asdict(result))
                    for m in msgs:
                        messages_fp.write(json.dumps({
                            'run_id': m.run_id, 'protocol': m.protocol,
                            'task_domain': m.task_domain, 'sender': m.sender,
                            'receiver': m.receiver, 'content': m.content,
                            'prompt_tokens': m.prompt_tokens,
                            'completion_tokens': m.completion_tokens,
                            'total_tokens': m.total_tokens,
                            'latency_ms': m.latency_ms,
                            'finish_reason': m.finish_reason,
                            'json_parse_error': m.json_parse_error,
                            'timestamp': m.timestamp,
                        }, ensure_ascii=False) + '\n')
                    messages_fp.flush()
                except Exception as e:
                    errors += 1
                    print(f'  ERROR {protocol.value}/{domain.value}/s{idx}/r{rep}: {e}')
                done += 1
                if done % 20 == 0:
                    print(f'  {done}/{N_TOTAL_RUNS} runs done ({errors} errors)...')
                time.sleep(0.3)

messages_fp.close()
df = pd.DataFrame(results)
print(f'\nDone: {len(df)} runs, {errors} errors')
print(f'Total tokens used: {df["total_tokens"].sum():,}')
print(f'Messages persisted to: {MESSAGES_PATH}')
df.head()

  20/360 runs done (0 errors)...
  40/360 runs done (0 errors)...
  60/360 runs done (0 errors)...
  80/360 runs done (0 errors)...
  100/360 runs done (0 errors)...
  120/360 runs done (0 errors)...
  140/360 runs done (0 errors)...
  160/360 runs done (0 errors)...
  180/360 runs done (0 errors)...
  200/360 runs done (0 errors)...
  220/360 runs done (0 errors)...
  240/360 runs done (0 errors)...
  260/360 runs done (0 errors)...
  280/360 runs done (0 errors)...
  300/360 runs done (0 errors)...
  320/360 runs done (0 errors)...
  340/360 runs done (0 errors)...
  360/360 runs done (0 errors)...

Done: 360 runs, 0 errors
Total tokens used: 396,150
Messages persisted to: results/results_messages.jsonl


,run_id,protocol,task_domain,sample_index,repetition,total_prompt_tokens,total_completion_tokens,total_tokens,total_latency_ms,total_messages,any_truncation,any_json_parse_error,completion_score,final_answer
0,94988703,NL,MATH,0,0,615,290,905,6931.00,3,False,False,1.0,Janet's ducks lay 16 eggs each day. After cons...
1,f1719d31,NL,MATH,0,1,602,255,857,6248.97,3,False,False,1.0,Janet's ducks lay 16 eggs each day. After cons...
2,f8a8632a,NL,MATH,0,2,651,310,961,5550.37,3,False,False,1.0,Janet's ducks lay 16 eggs each day. After usin...
3,80a938f9,NL,MATH,1,0,515,227,742,5059.21,3,False,False,1.0,The total amount of fiber needed for the robe ...
4,cfdc3e33,NL,MATH,1,1,550,262,812,6135.33,3,False,False,1.0,The total amount of fiber needed for the robe ...


In [10]:
df.to_csv('results/results_raw.csv', index=False)
print(f'Saved: results/results_raw.csv  ({len(df)} rows)')

Saved: results/results_raw.csv  (360 rows)


## 6. Data Quality Audit

Check truncation rate (`finish_reason == 'length'`) and JSON parse-failure rate
per cell. If either exceeds 5%, results for that cell should be interpreted
cautiously or re-run with a larger `DOMAIN_MAX_TOKENS`.

In [11]:
trunc_rate = df.groupby(['protocol', 'task_domain'])['any_truncation'].mean().round(3)
json_err_rate = df.groupby(['protocol', 'task_domain'])['any_json_parse_error'].mean().round(3)
print('Truncation rate (finish_reason=length) per cell:')
print(trunc_rate.to_string())
print('\nJSON parse-error rate per cell:')
print(json_err_rate.to_string())

hot = trunc_rate[trunc_rate > 0.05]
if len(hot):
    print(f'\nWARNING: {len(hot)} cells had >5% truncation — interpret with care.')
else:
    print('\nAll cells below 5% truncation threshold.')

Truncation rate (finish_reason=length) per cell:
protocol       task_domain
JSON           MATH           0.000
               NEWS           0.000
               READING        0.000
MARKDOWN       MATH           0.200
               NEWS           0.000
               READING        0.000
NL             MATH           0.067
               NEWS           0.000
               READING        0.000
SHARED_MEMORY  MATH           0.100
               NEWS           0.000
               READING        0.000

JSON parse-error rate per cell:
protocol       task_domain
JSON           MATH           0.0
               NEWS           0.0
               READING        0.0
MARKDOWN       MATH           0.0
               NEWS           0.0
               READING        0.0
NL             MATH           0.0
               NEWS           0.0
               READING        0.0
SHARED_MEMORY  MATH           0.0
               NEWS           0.0
               READING        0.0



## 7. RQ1 — Protocol Efficiency by Domain

Token cost and latency across protocols × domains. Two-way ANOVA on total_tokens
and latency, with protocol × domain interaction. Shared Memory is expected (H1)
to have the highest input tokens due to blackboard state accumulation.

In [12]:
eff = df.groupby(['protocol', 'task_domain']).agg(
    mean_tokens      = ('total_tokens',             'mean'),
    mean_prompt_tok  = ('total_prompt_tokens',      'mean'),
    mean_compl_tok   = ('total_completion_tokens',  'mean'),
    mean_latency     = ('total_latency_ms',         'mean'),
    n_runs           = ('run_id',                    'count'),
).round(1)

print('RQ1: Protocol Efficiency by Domain')
print(eff.to_string())

import statsmodels.api as sm
from statsmodels.formula.api import ols

print('\n--- Two-way ANOVA on total_tokens ---')
model_tok = ols('total_tokens ~ C(protocol) + C(task_domain) + C(protocol):C(task_domain)', data=df).fit()
anova_tok = sm.stats.anova_lm(model_tok, typ=2)
anova_tok['eta_sq'] = anova_tok['sum_sq'] / anova_tok['sum_sq'].sum()
print(anova_tok.round(4))

print('\n--- Two-way ANOVA on total_latency_ms ---')
model_lat = ols('total_latency_ms ~ C(protocol) + C(task_domain) + C(protocol):C(task_domain)', data=df).fit()
anova_lat = sm.stats.anova_lm(model_lat, typ=2)
anova_lat['eta_sq'] = anova_lat['sum_sq'] / anova_lat['sum_sq'].sum()
print(anova_lat.round(4))

RQ1: Protocol Efficiency by Domain
                           mean_tokens  mean_prompt_tok  mean_compl_tok  mean_latency  n_runs
protocol      task_domain                                                                    
JSON          MATH               846.7            599.5           247.2        5400.3      30
              NEWS              1313.1            815.0           498.2        9291.3      30
              READING            955.0            777.2           177.8        4659.9      30
MARKDOWN      MATH              1045.8            694.5           351.2        7414.1      30
              NEWS              1433.3            859.6           573.6        9576.0      30
              READING            994.7            794.0           200.7        4296.5      30
NL            MATH               987.8            664.5           323.3        6375.2      30
              NEWS               995.2            679.8           315.4        6176.1      30
              READING    

## 8. RQ2 — Completion Score × Protocol × Domain

Per-domain one-way ANOVA is the primary inferential test (see proposal §6.2)
because the `completion_score` metric is binary for Math but continuous for
Reading and News — a pooled two-way ANOVA mixes metric scales. The two-way
ANOVA is reported for completeness.

In [13]:
from itertools import combinations

completion = df.groupby(['protocol', 'task_domain']).agg(
    mean_score = ('completion_score', 'mean'),
    std_score  = ('completion_score',  'std'),
    n_runs     = ('run_id',           'count'),
).round(3)
print('RQ2: Completion — Protocol × Domain')
print(completion.to_string())

print('\n--- Two-way ANOVA on completion_score (exploratory — mixed metric scales) ---')
model_c = ols('completion_score ~ C(protocol) + C(task_domain) + C(protocol):C(task_domain)', data=df).fit()
anova_c = sm.stats.anova_lm(model_c, typ=2)
anova_c['eta_sq'] = anova_c['sum_sq'] / anova_c['sum_sq'].sum()
print(anova_c.round(4))

print('\n--- Per-domain one-way ANOVA + pairwise Cohen d ---')
for dom in TaskDomain:
    dom_df = df[df['task_domain'] == dom.value]
    groups = {p.value: dom_df[dom_df['protocol'] == p.value]['completion_score'].values for p in Protocol}
    glist = list(groups.values())
    f, pval = stats.f_oneway(*glist)
    grand = dom_df['completion_score'].mean()
    ss_b = sum(len(g) * (g.mean() - grand) ** 2 for g in glist)
    ss_t = sum((dom_df['completion_score'] - grand) ** 2)
    eta2 = ss_b / ss_t if ss_t > 0 else 0

    print(f'\n  [{dom.value}] F={f:.3f}, p={pval:.4f}, eta^2={eta2:.3f}',
          '-> SIG' if pval < 0.05 else '-> ns')
    if pval < 0.05:
        for (n1, g1), (n2, g2) in combinations(groups.items(), 2):
            pooled = np.sqrt((g1.var() + g2.var()) / 2)
            d = (g1.mean() - g2.mean()) / pooled if pooled > 0 else 0
            print(f'    Cohen d ({n1} vs {n2}): {d:+.3f}')

RQ2: Completion — Protocol × Domain
                           mean_score  std_score  n_runs
protocol      task_domain                               
JSON          MATH              0.733      0.450      30
              NEWS              0.296      0.086      30
              READING           0.419      0.221      30
MARKDOWN      MATH              0.867      0.346      30
              NEWS              0.271      0.054      30
              READING           0.347      0.197      30
NL            MATH              0.833      0.379      30
              NEWS              0.275      0.047      30
              READING           0.426      0.208      30
SHARED_MEMORY MATH              0.833      0.379      30
              NEWS              0.346      0.050      30
              READING           0.556      0.273      30

--- Two-way ANOVA on completion_score (exploratory — mixed metric scales) ---
                             sum_sq     df         F  PR(>F)  eta_sq
C(protocol)       

## 9. Tukey HSD Post-hoc

When an omnibus ANOVA is significant, Tukey HSD identifies *which* protocol
pairs drive the effect, with family-wise error control.

In [14]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

print('--- Tukey HSD on token cost (pooled across domains) ---')
print(pairwise_tukeyhsd(df['total_tokens'], df['protocol'], alpha=0.05))

print('\n--- Tukey HSD on completion_score, per domain ---')
for dom in TaskDomain:
    sub = df[df['task_domain'] == dom.value]
    print(f'\n[{dom.value}]')
    print(pairwise_tukeyhsd(sub['completion_score'], sub['protocol'], alpha=0.05))

--- Tukey HSD on token cost (pooled across domains) ---
       Multiple Comparison of Means - Tukey HSD, FWER=0.05        
 group1      group2     meandiff p-adj    lower     upper   reject
------------------------------------------------------------------
    JSON      MARKDOWN  119.6556 0.0012    37.409  201.9021   True
    JSON            NL -104.3444 0.0064  -186.591  -22.0979   True
    JSON SHARED_MEMORY  233.2889    0.0  151.0423  315.5355   True
MARKDOWN            NL    -224.0    0.0 -306.2466 -141.7534   True
MARKDOWN SHARED_MEMORY  113.6333 0.0023   31.3868  195.8799   True
      NL SHARED_MEMORY  337.6333    0.0  255.3868  419.8799   True
------------------------------------------------------------------

--- Tukey HSD on completion_score, per domain ---

[MATH]
    Multiple Comparison of Means - Tukey HSD, FWER=0.05     
 group1      group2    meandiff p-adj   lower  upper  reject
------------------------------------------------------------
    JSON      MARKDOWN   0.1333 

## 10. Bootstrap 95% Confidence Intervals

In [15]:
def bootstrap_ci(data, n_boot=2000, alpha=0.05, seed=42):
    rng = np.random.RandomState(seed)
    boot = [np.mean(rng.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
    return round(np.percentile(boot, 100 * alpha / 2), 3), round(np.percentile(boot, 100 * (1 - alpha / 2)), 3)

print('=== Token usage 95% CI (pooled) ===')
for p in Protocol:
    d = df[df['protocol'] == p.value]['total_tokens'].values
    lo, hi = bootstrap_ci(d)
    print(f'  {p.value:<14} mean={np.mean(d):7.1f}  95% CI [{lo}, {hi}]')

print('\n=== Completion score 95% CI (per domain) ===')
for dom in TaskDomain:
    print(f'\n  [{dom.value}]')
    for p in Protocol:
        d = df[(df['protocol'] == p.value) & (df['task_domain'] == dom.value)]['completion_score'].values
        if len(d) > 1:
            lo, hi = bootstrap_ci(d)
            print(f'    {p.value:<14} mean={np.mean(d):.3f}  95% CI [{lo}, {hi}]')
        else:
            print(f'    {p.value:<14} mean={np.mean(d):.3f}  (too few samples)')

=== Token usage 95% CI (pooled) ===
  NL             mean=  933.9  95% CI [906.888, 963.738]
  MARKDOWN       mean= 1157.9  95% CI [1106.474, 1213.16]
  JSON           mean= 1038.3  95% CI [991.294, 1088.81]
  SHARED_MEMORY  mean= 1271.6  95% CI [1234.832, 1309.16]

=== Completion score 95% CI (per domain) ===

  [MATH]
    NL             mean=0.833  95% CI [0.7, 0.967]
    MARKDOWN       mean=0.867  95% CI [0.733, 0.967]
    JSON           mean=0.733  95% CI [0.567, 0.867]
    SHARED_MEMORY  mean=0.833  95% CI [0.7, 0.967]

  [READING]
    NL             mean=0.426  95% CI [0.355, 0.498]
    MARKDOWN       mean=0.347  95% CI [0.279, 0.414]
    JSON           mean=0.419  95% CI [0.342, 0.499]
    SHARED_MEMORY  mean=0.556  95% CI [0.46, 0.658]

  [NEWS]
    NL             mean=0.275  95% CI [0.259, 0.292]
    MARKDOWN       mean=0.271  95% CI [0.252, 0.291]
    JSON           mean=0.296  95% CI [0.267, 0.327]
    SHARED_MEMORY  mean=0.346  95% CI [0.329, 0.363]


## 11. Final Summary Table — Best Protocol per Domain

In [16]:
rows = []
for p in Protocol:
    for dom in TaskDomain:
        sub = df[(df['protocol'] == p.value) & (df['task_domain'] == dom.value)]
        rows.append({
            'Protocol': p.value,
            'Domain':   dom.value,
            'Mean Tokens':      round(sub['total_tokens'].mean(), 1),
            'Mean Prompt Tok':  round(sub['total_prompt_tokens'].mean(), 1),
            'Mean Compl Tok':   round(sub['total_completion_tokens'].mean(), 1),
            'Mean Latency (ms)': round(sub['total_latency_ms'].mean(), 1),
            'Completion Rate':  round(sub['completion_score'].mean(), 3),
            'N':                len(sub),
        })

summary = pd.DataFrame(rows)
print('FINAL SUMMARY TABLE')
print(summary.to_string(index=False))
summary.to_csv('results/results_summary.csv', index=False)
print('\nSaved: results/results_summary.csv')

print('\n=== Best protocol per domain (by completion rate, tiebreaker = fewer tokens) ===')
for dom in TaskDomain:
    sub = summary[summary['Domain'] == dom.value]
    best = sub.sort_values(['Completion Rate', 'Mean Tokens'], ascending=[False, True]).iloc[0]
    print(f'  {dom.value:<10} -> {best["Protocol"]:<14} (score={best["Completion Rate"]:.3f}, tokens={best["Mean Tokens"]:.0f})')

FINAL SUMMARY TABLE
     Protocol  Domain  Mean Tokens  Mean Prompt Tok  Mean Compl Tok  Mean Latency (ms)  Completion Rate  N
           NL    MATH        987.8            664.5           323.3             6375.2            0.833 30
           NL READING        818.7            717.2           101.5             3153.9            0.426 30
           NL    NEWS        995.2            679.8           315.4             6176.1            0.275 30
     MARKDOWN    MATH       1045.8            694.5           351.2             7414.1            0.867 30
     MARKDOWN READING        994.7            794.0           200.7             4296.5            0.347 30
     MARKDOWN    NEWS       1433.3            859.6           573.6             9576.0            0.271 30
         JSON    MATH        846.7            599.5           247.2             5400.3            0.733 30
         JSON READING        955.0            777.2           177.8             4659.9            0.419 30
         JSON    

## 12. Figures

All saved to `figures/` at 150 DPI. These drive the Results section of the
final report and the Technical Depth portion of the Demo.

In [17]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('figures', exist_ok=True)
sns.set_context('paper')
sns.set_style('whitegrid')
PROTOS = [p.value for p in Protocol]
DOMS = [d.value for d in TaskDomain]
PALETTE = sns.color_palette('Set2', n_colors=len(PROTOS))

def save_fig(name):
    path = f'figures/{name}.png'
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  saved {path}')

In [18]:
# Fig 1 — Total tokens by protocol x domain, with bootstrap 95% CI
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(DOMS))
width = 0.2
for i, p in enumerate(PROTOS):
    means, los, his = [], [], []
    for d in DOMS:
        vals = df[(df['protocol'] == p) & (df['task_domain'] == d)]['total_tokens'].values
        m = np.mean(vals)
        lo, hi = bootstrap_ci(vals)
        means.append(m); los.append(m - lo); his.append(hi - m)
    ax.bar(x + i * width - 1.5 * width, means, width, label=p,
           yerr=[los, his], color=PALETTE[i], capsize=3)
ax.set_xticks(x); ax.set_xticklabels(DOMS)
ax.set_ylabel('Mean total tokens per run')
ax.set_title('Fig 1 — Token cost by protocol x domain (bootstrap 95% CI)')
ax.legend(title='Protocol', loc='upper left')
save_fig('fig1_tokens_by_protocol_domain')

  saved figures/fig1_tokens_by_protocol_domain.png


In [19]:
# Fig 2 — Completion score distribution by cell (box plot)
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x='task_domain', y='completion_score', hue='protocol',
            order=DOMS, hue_order=PROTOS, palette=PALETTE, ax=ax)
ax.set_ylabel('Completion score')
ax.set_xlabel('Task domain')
ax.set_title('Fig 2 — Completion score by protocol x domain')
ax.legend(title='Protocol', bbox_to_anchor=(1.02, 1), loc='upper left')
save_fig('fig2_completion_by_cell')

  saved figures/fig2_completion_by_cell.png


In [20]:
# Fig 3 — Interaction plot (H3 canonical visualization)
fig, ax = plt.subplots(figsize=(7, 5))
for i, p in enumerate(PROTOS):
    means = [df[(df['protocol'] == p) & (df['task_domain'] == d)]['completion_score'].mean() for d in DOMS]
    ax.plot(DOMS, means, marker='o', label=p, color=PALETTE[i], linewidth=2)
ax.set_xlabel('Task domain')
ax.set_ylabel('Mean completion score')
ax.set_title('Fig 3 — Protocol x Domain interaction (H3)')
ax.legend(title='Protocol')
save_fig('fig3_interaction')

  saved figures/fig3_interaction.png


In [21]:
# Fig 4 — Efficiency-effectiveness Pareto scatter
fig, ax = plt.subplots(figsize=(7, 5))
markers = {'MATH': 'o', 'READING': 's', 'NEWS': '^'}
for i, p in enumerate(PROTOS):
    for d in DOMS:
        sub = df[(df['protocol'] == p) & (df['task_domain'] == d)]
        if len(sub):
            ax.scatter(sub['total_tokens'].mean(), sub['completion_score'].mean(),
                       s=150, color=PALETTE[i], marker=markers[d],
                       edgecolor='black', linewidth=0.5)
ax.set_xlabel('Mean total tokens per run  (lower = cheaper)')
ax.set_ylabel('Mean completion score  (higher = better)')
ax.set_title('Fig 4 — Efficiency vs effectiveness trade-off')
proto_handles = [plt.Line2D([0],[0], marker='s', color=PALETTE[i], linestyle='', label=p, markersize=10) for i, p in enumerate(PROTOS)]
dom_handles = [plt.Line2D([0],[0], marker=markers[d], color='gray', linestyle='', label=d, markersize=10) for d in DOMS]
leg1 = ax.legend(handles=proto_handles, title='Protocol', loc='lower left')
ax.add_artist(leg1)
ax.legend(handles=dom_handles, title='Domain', loc='lower right')
save_fig('fig4_pareto')

  saved figures/fig4_pareto.png


In [22]:
# Fig 5 — Input vs output tokens, stacked by protocol
fig, ax = plt.subplots(figsize=(7, 5))
prompt_means = [df[df['protocol'] == p]['total_prompt_tokens'].mean() for p in PROTOS]
compl_means  = [df[df['protocol'] == p]['total_completion_tokens'].mean() for p in PROTOS]
ax.bar(PROTOS, prompt_means, label='Prompt (input)', color='#4C72B0')
ax.bar(PROTOS, compl_means, bottom=prompt_means, label='Completion (output)', color='#DD8452')
ax.set_ylabel('Mean tokens per run')
ax.set_title('Fig 5 — Input vs output token breakdown')
ax.legend()
save_fig('fig5_input_vs_output')

  saved figures/fig5_input_vs_output.png


In [23]:
# Fig 6 — Latency distribution by protocol
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x='protocol', y='total_latency_ms',
            order=PROTOS, palette=PALETTE, ax=ax)
ax.set_ylabel('Total latency per run (ms)')
ax.set_title('Fig 6 — Latency distribution by protocol')
save_fig('fig6_latency')

  saved figures/fig6_latency.png


## 13. Sample Outputs

First successful run per (protocol, domain) cell, for qualitative inspection.

In [24]:
for p in Protocol:
    for dom in TaskDomain:
        row = df[(df['protocol'] == p.value) & (df['task_domain'] == dom.value)]
        if len(row):
            r = row.iloc[0]
            print(f'[{p.value:<14} / {dom.value:<8}]  score={r["completion_score"]:.3f}  tokens={r["total_tokens"]}')
            print(f'  {r["final_answer"][:140]}')
            print()

[NL             / MATH    ]  score=1.000  tokens=905
  Janet's ducks lay 16 eggs each day. After consuming three eggs for breakfast and four for baking, she has seven eggs consumed daily. This le

[NL             / READING ]  score=0.400  tokens=820
  The Denver Broncos represented the AFC in Super Bowl 50.

[NL             / NEWS    ]  score=0.342  tokens=1038
  In March 2025, the Federal Reserve decided to maintain its benchmark interest rate at 4.25% to 4.50%. Chair Jerome Powell highlighted that i

[MARKDOWN       / MATH    ]  score=1.000  tokens=1027
  ## Final Answer
Janet makes **$18** every day at the farmers' market.

#### <18>

[MARKDOWN       / READING ]  score=0.333  tokens=1095
  ## Final Answer
- The Denver Broncos represented the AFC at Super Bowl 50.

[MARKDOWN       / NEWS    ]  score=0.259  tokens=1595
  ## Federal Reserve Interest Rate Decision - March 2025

### Key Information
- **Main Subject**: Federal Reserve's interest rate decision
- *

[JSON           / MATH  

## 14. Export Experiment Config for Reproducibility

In [25]:
config = {
    'project': 'Communication Protocol Effects on Multi-Agent System Efficiency',
    'course':  'GR5293 Generative AI using Large Language Models',
    'model':   MODEL,
    'temperature': 0.3,
    'domain_max_tokens': {k.value: v for k, v in DOMAIN_MAX_TOKENS.items()},
    'protocols': [p.value for p in Protocol],
    'domains':   [d.value for d in TaskDomain],
    'n_math_samples':    N_MATH,
    'n_reading_samples': N_READING,
    'n_news_samples':    N_NEWS,
    'n_reps':            N_REPS,
    'total_runs':        int(len(df)),
    'agent_architecture': {
        'agents':   ['Planning Agent', 'Execution Agent', 'Integration Agent'],
        'pipeline': 'Sequential: Planner -> Executor -> Integrator',
    },
    'evaluation': {
        'math':    'Numeric exact match (binary 0/1)',
        'reading': 'SQuAD token-level F1 vs gold answers (continuous 0-1)',
        'news':    'Mean of ROUGE-2 F1 and ROUGE-L F1 vs key-facts reference (continuous 0-1)',
    },
    'methodology_notes': {
        'shared_memory': 'Every agent is injected the full JSON snapshot of the blackboard; overhead scales with state size.',
        'nl_protocol':   'Explicit plain-English-prose instruction to prevent default-to-markdown drift.',
        'json_protocol': 'OpenAI response_format=json_object + one parse-retry.',
        'reproducibility': 'OpenAI seed parameter matches rep index; random / numpy seeded for sampling.',
    },
    'metrics': ['total_prompt_tokens', 'total_completion_tokens', 'total_tokens', 'total_latency_ms', 'completion_score'],
}

with open('results/experiment_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved: results/experiment_config.json')
print(json.dumps(config, indent=2))

Saved: results/experiment_config.json
{
  "project": "Communication Protocol Effects on Multi-Agent System Efficiency",
  "course": "GR5293 Generative AI using Large Language Models",
  "model": "gpt-4o-mini",
  "temperature": 0.3,
  "domain_max_tokens": {
    "MATH": 256,
    "READING": 256,
    "NEWS": 512
  },
  "protocols": [
    "NL",
    "MARKDOWN",
    "JSON",
    "SHARED_MEMORY"
  ],
  "domains": [
    "MATH",
    "READING",
    "NEWS"
  ],
  "n_math_samples": 10,
  "n_reading_samples": 10,
  "n_news_samples": 10,
  "n_reps": 3,
  "total_runs": 360,
  "agent_architecture": {
    "agents": [
      "Planning Agent",
      "Execution Agent",
      "Integration Agent"
    ],
    "pipeline": "Sequential: Planner -> Executor -> Integrator"
  },
  "evaluation": {
    "math": "Numeric exact match (binary 0/1)",
    "reading": "SQuAD token-level F1 vs gold answers (continuous 0-1)",
    "news": "Mean of ROUGE-2 F1 and ROUGE-L F1 vs key-facts reference (continuous 0-1)"
  },
  "methodolo